# Case Study 8 — Web Scraping to Vector Database

**Student:** Siddhartha Mukherjee  
**Date:** May 2026  

This notebook builds a full pipeline: **scrape → clean → chunk → embed → store → search** using ChromaDB.

## Pipeline Overview
1. Scrape text from 3 different websites using BeautifulSoup
2. Clean and process the raw HTML into plain text
3. Split text into smaller chunks (with overlap)
4. Generate vector embeddings using SentenceTransformers
5. Store embeddings in ChromaDB vector database
6. Run semantic search queries to find relevant content
7. Experiment with different chunk sizes and compare results

## Part 0 — Setup & Configuration

**Prerequisites:**
- Docker containers running: `chromadb-server`, `ollama`, `open-webui`
- Python virtual environment activated
- All packages from `requirements.txt` installed

**Ethical Note:** This project respects `robots.txt` policies. We use a polite 1-second delay between requests and only scrape publicly available content for educational purposes.

In [1]:
# ---- Part 0: Install Dependencies (run once, then comment out) ----
# %pip install python-dotenv beautifulsoup4 lxml sentence-transformers chromadb tqdm requests numpy pandas

In [2]:
# ---- Part 0: Imports ----
import os
import time
import re
import uuid
import hashlib
import requests
import numpy as np
from urllib.parse import urlparse
from datetime import datetime
from dotenv import load_dotenv
from bs4 import BeautifulSoup
from tqdm import tqdm
import chromadb
from chromadb.config import Settings
from sentence_transformers import SentenceTransformer

print("All libraries imported successfully!")

All libraries imported successfully!


In [3]:
# ---- Part 0: Load Configuration from .env ----
load_dotenv(override=True)

CHROMA_HOST = os.getenv("CHROMA_HOST", "localhost")
CHROMA_PORT = int(os.getenv("CHROMA_PORT", "8000"))
COLLECTION_NAME = os.getenv("COLLECTION_NAME", "web_scraping_collection")
EMBEDDING_MODEL = os.getenv("EMBEDDING_MODEL", "all-MiniLM-L6-v2")
SPACE_TYPE = os.getenv("SPACE_TYPE", "cosine")

# Default chunk settings (we will experiment with these later)
CHUNK_SIZE = int(os.getenv("CHUNK_SIZE", "1000"))
CHUNK_OVERLAP = int(os.getenv("CHUNK_OVERLAP", "200"))

print(f"Configuration:")
print(f"  ChromaDB: {CHROMA_HOST}:{CHROMA_PORT}")
print(f"  Collection: {COLLECTION_NAME}")
print(f"  Embedding Model: {EMBEDDING_MODEL}")
print(f"  Chunk Size: {CHUNK_SIZE}, Overlap: {CHUNK_OVERLAP}")
print(f"  Distance Metric: {SPACE_TYPE}")

Configuration:
  ChromaDB: localhost:8000
  Collection: web_scraping_collection
  Embedding Model: all-MiniLM-L6-v2
  Chunk Size: 1000, Overlap: 200
  Distance Metric: cosine


In [4]:
# ---- Part 0: Verify ChromaDB Connection ----
client = chromadb.HttpClient(
    host=CHROMA_HOST,
    port=CHROMA_PORT,
    settings=Settings(allow_reset=True, anonymized_telemetry=False)
)

heartbeat = client.heartbeat()
print(f"ChromaDB Heartbeat: {heartbeat}")
print("ChromaDB is running and connected!")

ChromaDB Heartbeat: 1779547453152556129
ChromaDB is running and connected!


## Part 1 — URL Selection & Web Scraping

### URL Selection Rationale

I chose 3 URLs from **different domains** with **different types of content**:

| # | URL | Domain | Why I Chose It |
|---|-----|--------|----------------|
| 1 | Wikipedia: Natural Language Processing | en.wikipedia.org | Long, structured educational article with lots of text about NLP — directly related to this course |
| 2 | Python Official Tutorial | docs.python.org | Technical documentation with code examples — tests how the system handles mixed content |
| 3 | Project Gutenberg: Alice's Adventures in Wonderland | www.gutenberg.org | Classic public domain literature — different writing style to test content diversity |

**Diversity:** Educational (Wikipedia) + Technical Docs (Python.org) + Literature (Gutenberg)  
**All are:** Publicly accessible, English, text-heavy, from different domains, and allow scraping.

In [5]:
# ---- Part 1: Define URLs ----
# Three URLs from different domains with different content types
URLS = [
    "https://en.wikipedia.org/wiki/Natural_language_processing",    # Educational
    "https://docs.python.org/3/tutorial/introduction.html",         # Technical Docs
    "https://www.gutenberg.org/cache/epub/11/pg11-images.html",     # Literature
]

print("URLs to scrape:")
for i, url in enumerate(URLS, 1):
    domain = urlparse(url).netloc
    print(f"  {i}. [{domain}] {url}")

URLs to scrape:
  1. [en.wikipedia.org] https://en.wikipedia.org/wiki/Natural_language_processing
  2. [docs.python.org] https://docs.python.org/3/tutorial/introduction.html
  3. [www.gutenberg.org] https://www.gutenberg.org/cache/epub/11/pg11-images.html


In [6]:
# ---- Part 1: Scraping Functions ----

def fetch_html(url, timeout=20):
    """Fetch raw HTML from a URL with a polite user-agent."""
    headers = {"User-Agent": "Mozilla/5.0 (Educational Project - Web Scraping Assignment)"}
    response = requests.get(url, headers=headers, timeout=timeout)
    response.raise_for_status()
    print(f"    Status: {response.status_code}, Size: {len(response.text):,} chars")
    return response.text


def extract_visible_text(html):
    """Extract only visible text from HTML, removing scripts, styles, nav, footer."""
    soup = BeautifulSoup(html, "lxml")
    
    # Remove non-content tags
    for tag in soup(["script", "style", "noscript", "header", "footer", "nav", "svg", "img"]):
        tag.decompose()
    
    # Get text with newline separators
    text = soup.get_text(separator="\n")
    
    # Clean up whitespace
    text = re.sub(r"\s+\n", "\n", text)
    text = re.sub(r"[ \t]+", " ", text)
    lines = [line.strip() for line in text.splitlines()]
    lines = [line for line in lines if line]  # Remove empty lines
    
    return "\n".join(lines)


def scrape_urls(urls, sleep_time=1.0):
    """Scrape a list of URLs with polite delay between requests.
    
    Respects rate limiting by waiting between requests.
    Note: Always check robots.txt before scraping any website.
    """
    documents = []
    
    for i, url in enumerate(urls, 1):
        print(f"\n  [{i}/{len(urls)}] Scraping: {url}")
        try:
            html = fetch_html(url)
            text = extract_visible_text(html)
            
            if len(text) < 500:
                print(f"    [Warning] Very short content: only {len(text)} chars")
            else:
                print(f"    Extracted: {len(text):,} chars of clean text")
            
            documents.append({
                "url": url,
                "domain": urlparse(url).netloc,
                "text": text,
                "scraped_at": datetime.now().isoformat()
            })
            
            # Polite delay between requests (rate limiting)
            if i < len(urls):
                print(f"    Waiting {sleep_time}s before next request (rate limiting)...")
                time.sleep(sleep_time)
                
        except Exception as e:
            print(f"    [Error] Failed to scrape {url}: {e}")
    
    return documents

print("Scraping functions defined!")

Scraping functions defined!


In [7]:
# ---- Part 1: Execute Scraping ----
print("Starting web scraping...")
print("(Respecting rate limits with 1s delay between requests)")

raw_docs = scrape_urls(URLS, sleep_time=1.0)

print(f"\n--- Scraping Summary ---")
print(f"Total documents scraped: {len(raw_docs)}")
for doc in raw_docs:
    print(f"  {doc['domain']}: {len(doc['text']):,} chars")

Starting web scraping...
(Respecting rate limits with 1s delay between requests)

  [1/3] Scraping: https://en.wikipedia.org/wiki/Natural_language_processing
    Status: 200, Size: 323,087 chars
    Extracted: 52,199 chars of clean text
    Waiting 1.0s before next request (rate limiting)...

  [2/3] Scraping: https://docs.python.org/3/tutorial/introduction.html
    Status: 200, Size: 73,158 chars
    Extracted: 18,248 chars of clean text
    Waiting 1.0s before next request (rate limiting)...

  [3/3] Scraping: https://www.gutenberg.org/cache/epub/11/pg11-images.html
    Status: 200, Size: 185,223 chars
    Extracted: 162,100 chars of clean text

--- Scraping Summary ---
Total documents scraped: 3
  en.wikipedia.org: 52,199 chars
  docs.python.org: 18,248 chars
  www.gutenberg.org: 162,100 chars


## Part 2 — Text Chunking

We split scraped text into smaller chunks so each chunk fits well for embedding.  
**Overlap** ensures context is not lost between chunks.

In [8]:
# ---- Part 2: Chunking Functions ----

def chunk_text(text, chunk_size=1000, chunk_overlap=200):
    """Split text into overlapping chunks of given size.
    
    Args:
        text: The input text to split
        chunk_size: Max characters per chunk
        chunk_overlap: Number of overlapping characters between chunks
    
    Returns:
        List of text chunks
    """
    chunks = []
    start = 0
    
    while start < len(text):
        end = start + chunk_size
        chunk = text[start:end]
        
        if chunk.strip():  # Only add non-empty chunks
            chunks.append(chunk.strip())
        
        start = end - chunk_overlap
        if start >= len(text):
            break
    
    return chunks


def build_corpus(docs, chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP):
    """Build a corpus of chunks from scraped documents with metadata."""
    corpus = []
    
    for doc in docs:
        url = doc["url"]
        domain = doc["domain"]
        chunks = chunk_text(doc["text"], chunk_size=chunk_size, chunk_overlap=chunk_overlap)
        
        for i, chunk in enumerate(chunks):
            # Create a unique ID using hash
            doc_id = hashlib.sha256(chunk.encode('utf-8')).hexdigest()[:16]
            
            corpus.append({
                "id": f"{domain}-{i}-{doc_id}",
                "text": chunk,
                "metadata": {
                    "source": url,
                    "domain": domain,
                    "chunk_index": i,
                    "length": len(chunk),
                    "timestamp": doc.get("scraped_at", datetime.now().isoformat())
                }
            })
    
    return corpus

print("Chunking functions defined!")

Chunking functions defined!


In [9]:
# ---- Part 2: Execute Chunking ----
corpus = build_corpus(raw_docs, chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP)

print(f"--- Chunking Summary ---")
print(f"Chunk Size: {CHUNK_SIZE}, Overlap: {CHUNK_OVERLAP}")
print(f"Total chunks created: {len(corpus)}")

# Show per-domain breakdown
from collections import Counter
domain_counts = Counter(item["metadata"]["domain"] for item in corpus)
for domain, count in domain_counts.items():
    print(f"  {domain}: {count} chunks")

# Verify we have >= 50 chunks (rubric requirement)
if len(corpus) >= 50:
    print(f"\nMeets requirement: {len(corpus)} chunks (>= 50 required)")
else:
    print(f"\n[Warning] Only {len(corpus)} chunks. Need at least 50.")

--- Chunking Summary ---
Chunk Size: 1000, Overlap: 200
Total chunks created: 292
  en.wikipedia.org: 66 chunks
  docs.python.org: 23 chunks
  www.gutenberg.org: 203 chunks

Meets requirement: 292 chunks (>= 50 required)


## Part 3 — Embeddings & ChromaDB Storage

We now:
1. Load the SentenceTransformer model (`all-MiniLM-L6-v2`)
2. Convert each text chunk into a vector embedding
3. Store everything in ChromaDB with metadata

In [10]:
# ---- Part 3: Load Embedding Model ----
print(f"Loading embedding model: {EMBEDDING_MODEL}")
model = SentenceTransformer(EMBEDDING_MODEL)

# Show model info
test_embedding = model.encode(["test"])
print(f"Model loaded successfully!")
print(f"Embedding dimensions: {len(test_embedding[0])}")
print(f"This means each text chunk becomes a vector of {len(test_embedding[0])} numbers.")

Loading embedding model: all-MiniLM-L6-v2


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Model loaded successfully!
Embedding dimensions: 384
This means each text chunk becomes a vector of 384 numbers.


In [11]:
# ---- Part 3: Show What an Embedding Looks Like ----
sample_text = "Natural language processing helps computers understand human text."
sample_embedding = model.encode([sample_text], normalize_embeddings=(SPACE_TYPE == "cosine"))

print(f"Sample text: \"{sample_text}\"")
print(f"Embedding shape: {sample_embedding.shape}")
print(f"Embedding dimensions: {len(sample_embedding[0])}")
print(f"\nFirst 20 values of the embedding vector:")
print(sample_embedding[0][:20])
print(f"\nFull embedding vector (truncated display):")
print(sample_embedding[0])

Sample text: "Natural language processing helps computers understand human text."
Embedding shape: (1, 384)
Embedding dimensions: 384

First 20 values of the embedding vector:
[ 0.02582469  0.03353431  0.06706765 -0.0494164   0.01574439 -0.053859
  0.06529717  0.00594689  0.0820811   0.00713835 -0.01050178  0.03082515
  0.03053222 -0.01988104  0.10354853  0.02185947 -0.03756901  0.00165901
 -0.0945642  -0.07407047]

Full embedding vector (truncated display):
[ 2.58246940e-02  3.35343145e-02  6.70676455e-02 -4.94163968e-02
  1.57443918e-02 -5.38589954e-02  6.52971715e-02  5.94689324e-03
  8.20811018e-02  7.13835377e-03 -1.05017833e-02  3.08251530e-02
  3.05322185e-02 -1.98810417e-02  1.03548527e-01  2.18594745e-02
 -3.75690050e-02  1.65900867e-03 -9.45641994e-02 -7.40704685e-02
  2.57691760e-02  4.35606390e-02 -2.81226691e-02 -1.55006200e-02
  7.83475116e-03  9.55023840e-02 -5.24337329e-02 -8.00070316e-02
  1.03690177e-01 -5.65181486e-03  2.54225563e-02  1.52936755e-02
  1.45645216e-01 

In [26]:
# ---- Part 3: Create/Reset ChromaDB Collection ----

# Delete old collection if it exists (clean start)
try:
    client.delete_collection(name=COLLECTION_NAME)
    print(f"Deleted old collection: {COLLECTION_NAME}")
except Exception:
    print(f"No old collection to delete. Starting fresh.")

# Create new collection
collection = client.create_collection(
    name=COLLECTION_NAME,
    metadata={"hnsw:space": SPACE_TYPE}
)
print(f"Created collection: {COLLECTION_NAME} (distance: {SPACE_TYPE})")

Deleted old collection: web_scraping_collection
Created collection: web_scraping_collection (distance: cosine)


In [13]:
# ---- Part 3: Generate Embeddings & Store in ChromaDB ----

def embed_and_store(corpus_items, collection, model, batch_size=32):
    """Generate embeddings and store in ChromaDB in batches."""
    total = len(corpus_items)
    stored = 0
    
    print(f"Processing {total} chunks in batches of {batch_size}...")
    
    for i in range(0, total, batch_size):
        batch = corpus_items[i:i+batch_size]
        
        ids = [item["id"] for item in batch]
        texts = [item["text"] for item in batch]
        metadatas = [item["metadata"] for item in batch]
        
        # Generate embeddings for this batch
        embeddings = model.encode(
            texts, 
            normalize_embeddings=(SPACE_TYPE == "cosine")
        ).tolist()
        
        # Store in ChromaDB
        collection.add(
            ids=ids,
            documents=texts,
            metadatas=metadatas,
            embeddings=embeddings
        )
        
        stored += len(batch)
        print(f"  Stored {stored}/{total} chunks...")
    
    return stored

stored_count = embed_and_store(corpus, collection, model)
print(f"\nDone! Stored {stored_count} chunks in ChromaDB.")

Processing 292 chunks in batches of 32...
  Stored 32/292 chunks...
  Stored 64/292 chunks...
  Stored 96/292 chunks...
  Stored 128/292 chunks...
  Stored 160/292 chunks...
  Stored 192/292 chunks...
  Stored 224/292 chunks...
  Stored 256/292 chunks...
  Stored 288/292 chunks...
  Stored 292/292 chunks...

Done! Stored 292 chunks in ChromaDB.


In [14]:
# ---- Part 3: Verify Storage ----
count = collection.count()
print(f"--- ChromaDB Collection Status ---")
print(f"  Collection Name: {COLLECTION_NAME}")
print(f"  Total Documents: {count}")
print(f"  Distance Metric: {SPACE_TYPE}")
print(f"  Embedding Dimensions: {len(test_embedding[0])}")

# Show sample documents
if count > 0:
    sample = collection.peek(limit=3)
    print(f"\n--- Sample Documents ---")
    for i in range(min(3, len(sample['ids']))):
        doc_id = sample['ids'][i]
        meta = sample['metadatas'][i]
        doc_text = sample['documents'][i]
        print(f"\n  Document {i+1}:")
        print(f"    ID: {doc_id[:30]}...")
        print(f"    Source: {meta.get('source', 'Unknown')}")
        print(f"    Domain: {meta.get('domain', 'Unknown')}")
        print(f"    Length: {meta.get('length', 'Unknown')} chars")
        print(f"    Preview: {doc_text[:120]}...")

--- ChromaDB Collection Status ---
  Collection Name: web_scraping_collection
  Total Documents: 292
  Distance Metric: cosine
  Embedding Dimensions: 384

--- Sample Documents ---

  Document 1:
    ID: en.wikipedia.org-0-f834e3f1f82...
    Source: https://en.wikipedia.org/wiki/Natural_language_processing
    Domain: en.wikipedia.org
    Length: 999 chars
    Preview: Natural language processing - Wikipedia
Jump to content
From Wikipedia, the free encyclopedia
Processing of natural lang...

  Document 2:
    ID: en.wikipedia.org-1-b4be3fa8d95...
    Source: https://en.wikipedia.org/wiki/Natural_language_processing
    Domain: en.wikipedia.org
    Length: 1000 chars
    Preview: ggestions.
(
July 2025
)
This article
may be in need of reorganization to comply with Wikipedia's
layout guidelines
.
Pl...

  Document 3:
    ID: en.wikipedia.org-2-a472a8a7ff2...
    Source: https://en.wikipedia.org/wiki/Natural_language_processing
    Domain: en.wikipedia.org
    Length: 999 chars
    Previe

## Part 4 — Semantic Search

Now we test if the system actually understands meaning.  
We run **3 different search queries** — each designed to match content from a different source:

1. A **contextual** query about NLP (should match Wikipedia)
2. A **technical** query about programming (should match Python docs)
3. A **thematic** query about storytelling (should match Gutenberg literature)

We also compare **contextual vs keyword** style queries.

In [15]:
# ---- Part 4: Semantic Search Function ----

def semantic_search(query, collection, model, k=5):
    """Run a semantic search query and return results."""
    # Generate embedding for the query
    query_embedding = model.encode(
        [query], 
        normalize_embeddings=(SPACE_TYPE == "cosine")
    ).tolist()
    
    # Search ChromaDB
    results = collection.query(
        query_embeddings=query_embedding,
        n_results=k,
        include=["documents", "metadatas", "distances"]
    )
    
    return results


def display_results(query, results):
    """Print search results in a readable format."""
    print(f"\n{'='*70}")
    print(f"Query: \"{query}\"")
    print(f"{'='*70}")
    
    for i, (doc, meta, dist) in enumerate(zip(
        results["documents"][0],
        results["metadatas"][0],
        results["distances"][0]
    )):
        print(f"\n  Result {i+1} (Distance: {dist:.4f}):")
        print(f"    Domain: {meta.get('domain', 'Unknown')}")
        print(f"    Source: {meta.get('source', 'Unknown')}")
        print(f"    Preview: {doc[:180].replace(chr(10), ' ')}...")

print("Search functions defined!")

Search functions defined!


In [16]:
# ---- Part 4: Run 3 Semantic Search Queries ----

# Query 1: Contextual query — should find NLP-related content from Wikipedia
query1 = "How do machines understand and process human language?"
results1 = semantic_search(query1, collection, model, k=5)
display_results(query1, results1)


Query: "How do machines understand and process human language?"

  Result 1 (Distance: 0.4555):
    Domain: en.wikipedia.org
    Source: https://en.wikipedia.org/wiki/Natural_language_processing
    Preview: pects of semantics are concerned, and due to the development of powerful neural language models such as GPT-2 , this can now (2019) be considered a largely solved problem and is be...

  Result 2 (Distance: 0.4669):
    Domain: en.wikipedia.org
    Source: https://en.wikipedia.org/wiki/Natural_language_processing
    Preview: token along the sequence of N tokens PF is the probability function specific to a language Ties with cognitive linguistics are part of the historical heritage of NLP, but they have...

  Result 3 (Distance: 0.4750):
    Domain: en.wikipedia.org
    Source: https://en.wikipedia.org/wiki/Natural_language_processing
    Preview: y a computer . NLP is a subfield of computer science and is closely associated with artificial intelligence . NLP is also related to in

In [17]:
# Query 2: Technical query — should find programming content from Python docs
query2 = "How to use strings and numbers in Python programming"
results2 = semantic_search(query2, collection, model, k=5)
display_results(query2, results2)


Query: "How to use strings and numbers in Python programming"

  Result 1 (Distance: 0.3506):
    Domain: docs.python.org
    Source: https://docs.python.org/3/tutorial/introduction.html
    Preview: vior. In addition to int and float , Python supports other types of numbers, such as Decimal and Fraction . Python also has built-in support for complex numbers , and uses the j or...

  Result 2 (Distance: 0.4129):
    Domain: docs.python.org
    Source: https://docs.python.org/3/tutorial/introduction.html
    Preview: e 1 , in <module> TypeError : 'str' object does not support item assignment >>> word [ 2 :] = 'py' Traceback (most recent call last): File "<stdin>" , line 1 , in <module> TypeErro...

  Result 3 (Distance: 0.4345):
    Domain: docs.python.org
    Source: https://docs.python.org/3/tutorial/introduction.html
    Preview: a , end = ',' ) ... a , b = b , a + b ... 0,1,1,2,3,5,8,13,21,34,55,89,144,233,377,610,987, Footnotes [ 1 ] Since ** has higher precedence than - , -3**2 w

In [22]:
# Query 3: Thematic query — should find literature content from Gutenberg
query3 = "a curious girl falling down a rabbit hole into a strange world"
results3 = semantic_search(query3, collection, model, k=5)
display_results(query3, results3)


Query: "a curious girl falling down a rabbit hole into a strange world"

  Result 1 (Distance: 0.3480):
    Domain: www.gutenberg.org
    Source: https://www.gutenberg.org/cache/epub/11/pg11-images.html
    Preview: its waistcoat-pocket , and looked at it, and then hurried on, Alice started to her feet, for it flashed across her mind that she had never before seen a rabbit with either a waistc...

  Result 2 (Distance: 0.5011):
    Domain: www.gutenberg.org
    Source: https://www.gutenberg.org/cache/epub/11/pg11-images.html
    Preview: girl like you,” (she might well say this), “to go on crying in this way! Stop this moment, I tell you!” But she went on all the same, shedding gallons of tears, until there was a l...

  Result 3 (Distance: 0.5170):
    Domain: www.gutenberg.org
    Source: https://www.gutenberg.org/cache/epub/11/pg11-images.html
    Preview: ay ‘Who am I then? Tell me that first, and then, if I like being that person, I’ll come up: if not, I’ll stay down here till I’

In [23]:
# Bonus: Compare contextual vs keyword query
print("\n" + "="*70)
print("COMPARISON: Contextual Query vs Keyword Query")
print("="*70)

# Contextual (natural language)
q_contextual = "What are the real-world applications of understanding text with AI?"
r_contextual = semantic_search(q_contextual, collection, model, k=3)
print(f"\nContextual: \"{q_contextual}\"")
for i, (meta, dist) in enumerate(zip(r_contextual["metadatas"][0], r_contextual["distances"][0])):
    print(f"  {i+1}. [{meta['domain']}] Distance: {dist:.4f}")

# Keyword style
q_keyword = "NLP applications text"
r_keyword = semantic_search(q_keyword, collection, model, k=3)
print(f"\nKeyword: \"{q_keyword}\"")
for i, (meta, dist) in enumerate(zip(r_keyword["metadatas"][0], r_keyword["distances"][0])):
    print(f"  {i+1}. [{meta['domain']}] Distance: {dist:.4f}")

print("\nObservation: Contextual queries often find more relevant results")
print("because the embedding model understands meaning, not just keywords.")


COMPARISON: Contextual Query vs Keyword Query

Contextual: "What are the real-world applications of understanding text with AI?"
  1. [en.wikipedia.org] Distance: 0.4136
  2. [en.wikipedia.org] Distance: 0.4354
  3. [en.wikipedia.org] Distance: 0.4436

Keyword: "NLP applications text"
  1. [en.wikipedia.org] Distance: 0.4445
  2. [en.wikipedia.org] Distance: 0.4712
  3. [en.wikipedia.org] Distance: 0.4735

Observation: Contextual queries often find more relevant results
because the embedding model understands meaning, not just keywords.


## Part 5 — Experiments: Chunk Size Comparison

**Rubric Section B** asks for comparison of chunk sizes or model choices.  
Below we test 3 different chunk sizes and see how they affect:
- Number of chunks created
- Search result quality (distance scores)

In [24]:
# ---- Part 5: Experiment with Different Chunk Sizes ----

chunk_configs = [
    {"size": 500, "overlap": 100, "label": "Small (500/100)"},
    {"size": 1000, "overlap": 200, "label": "Medium (1000/200)"},
    {"size": 2000, "overlap": 400, "label": "Large (2000/400)"},
]

test_query = "How do machines understand human language?"
print(f"Test query: \"{test_query}\"")
print(f"\n{'Label':<22} {'Chunks':<10} {'Avg Distance':<15} {'Best Distance':<15}")
print("-" * 62)

for config in chunk_configs:
    # Build corpus with this chunk size
    temp_corpus = build_corpus(raw_docs, chunk_size=config["size"], chunk_overlap=config["overlap"])
    
    # Create a temporary collection
    temp_name = f"experiment_{config['size']}"
    try:
        client.delete_collection(name=temp_name)
    except Exception:
        pass
    
    temp_collection = client.create_collection(
        name=temp_name,
        metadata={"hnsw:space": SPACE_TYPE}
    )
    
    # Store embeddings
    for i in range(0, len(temp_corpus), 32):
        batch = temp_corpus[i:i+32]
        ids = [item["id"] for item in batch]
        texts = [item["text"] for item in batch]
        metas = [item["metadata"] for item in batch]
        embs = model.encode(texts, normalize_embeddings=(SPACE_TYPE=="cosine")).tolist()
        temp_collection.add(ids=ids, documents=texts, metadatas=metas, embeddings=embs)
    
    # Search
    q_emb = model.encode([test_query], normalize_embeddings=(SPACE_TYPE=="cosine")).tolist()
    results = temp_collection.query(query_embeddings=q_emb, n_results=5, include=["distances"])
    
    distances = results["distances"][0]
    avg_dist = np.mean(distances)
    best_dist = min(distances)
    
    print(f"{config['label']:<22} {len(temp_corpus):<10} {avg_dist:<15.4f} {best_dist:<15.4f}")
    
    # Clean up temporary collection
    client.delete_collection(name=temp_name)

print("\n--- Analysis ---")
print("Smaller chunks = more chunks but each chunk is more focused.")
print("Larger chunks = fewer chunks but each has more context.")
print("Medium (1000/200) usually gives a good balance for most use cases.")

Test query: "How do machines understand human language?"

Label                  Chunks     Avg Distance    Best Distance  
--------------------------------------------------------------
Small (500/100)        583        0.4854          0.4651         
Medium (1000/200)      292        0.4901          0.4680         
Large (2000/400)       147        0.5245          0.5014         

--- Analysis ---
Smaller chunks = more chunks but each chunk is more focused.
Larger chunks = fewer chunks but each has more context.
Medium (1000/200) usually gives a good balance for most use cases.


## Part 6 — Summary & Results

### Pipeline Summary
1. **Scraped** 3 websites from different domains using BeautifulSoup
2. **Cleaned** raw HTML by removing scripts, styles, and navigation
3. **Chunked** text into ~1000-char pieces with 200-char overlap
4. **Embedded** each chunk using `all-MiniLM-L6-v2` (384 dimensions)
5. **Stored** everything in ChromaDB with metadata (source, domain, length)
6. **Searched** using 3 different queries — all returned relevant results
7. **Compared** 3 chunk sizes to understand their effect on search quality

### Key Findings
- Semantic search works well: queries about NLP correctly find Wikipedia content, programming queries find Python docs
- Contextual queries (natural language) often produce better results than simple keyword queries
- Medium chunk size (1000 chars) gives a good balance between detail and context
- The cosine distance metric works well for normalized embeddings

In [25]:
# ---- Part 6: Final Verification ----
print("=" * 50)
print("FINAL STATUS")
print("=" * 50)
print(f"ChromaDB Heartbeat: {client.heartbeat()}")
print(f"Collection: {COLLECTION_NAME}")
print(f"Documents Stored: {collection.count()}")
print(f"Embedding Model: {EMBEDDING_MODEL}")
print(f"Embedding Dimensions: {len(test_embedding[0])}")
print(f"Chunk Size: {CHUNK_SIZE}, Overlap: {CHUNK_OVERLAP}")
print(f"Distance Metric: {SPACE_TYPE}")
print(f"URLs Scraped: {len(URLS)}")
for url in URLS:
    print(f"  - {url}")
print("\nPipeline completed successfully!")

FINAL STATUS
ChromaDB Heartbeat: 1779547596267473459
Collection: web_scraping_collection
Documents Stored: 292
Embedding Model: all-MiniLM-L6-v2
Embedding Dimensions: 384
Chunk Size: 1000, Overlap: 200
Distance Metric: cosine
URLs Scraped: 3
  - https://en.wikipedia.org/wiki/Natural_language_processing
  - https://docs.python.org/3/tutorial/introduction.html
  - https://www.gutenberg.org/cache/epub/11/pg11-images.html

Pipeline completed successfully!
